# Entrenamiento y exportación — Clasificador de enfermedades en plantas

Notebook para Google Colab (GPU). Descarga PlantVillage, entrena MobileNetV2,
genera las gráficas y exporta el modelo a ExecuTorch (`.pte`).

**Antes de empezar:** activa la GPU en `Entorno de ejecución > Cambiar tipo de entorno`.


## 1. Clonar el repositorio


In [ ]:
# El repositorio debe existir y ser público en GitHub con este nombre exacto
!git clone https://github.com/JabesMonroy/plant-disease-classifier.git proyecto
%cd proyecto

## 2. Instalar dependencias
Se fijan las versiones de torch/executorch para que el `.pte` sea compatible con el Space.


In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 executorch==0.4.0 \
    scikit-learn matplotlib pillow tqdm kaggle

## 3. Descargar el dataset PlantVillage
Sube tu archivo `kaggle.json` (Account > Create New API Token en Kaggle).


In [ ]:
from google.colab import files
files.upload()  # selecciona kaggle.json

In [ ]:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p data --unzip

## 4. Entrenar el modelo
Genera `artifacts/best.pt`, las gráficas y `labels.json`.


In [ ]:
!python src/train.py \
    --data-dir "data/plantvillage dataset/color" \
    --out-dir artifacts \
    --epochs 10 \
    --batch-size 64

## 5. Ver las gráficas de entrenamiento


In [ ]:
from IPython.display import Image, display
for img in ['curva_perdida.png','curva_accuracy.png','matriz_confusion.png']:
    display(Image(filename=f'artifacts/{img}'))

## 6. Exportar a ExecuTorch
Crea `space/model.pte` y copia `labels.json` junto a él.


In [ ]:
!python export/export_executorch.py \
    --checkpoint artifacts/best.pt \
    --labels artifacts/labels.json \
    --output space/model.pte

## 7. Descargar los archivos para el Space


In [ ]:
from google.colab import files
files.download('space/model.pte')
files.download('space/labels.json')